# Lernkurven, Auflösungsvermögen und Rauschgrenze

Arbeitsplan Schritt 5, Punkt 5. Dieses Notebook **enthält keine Logik** — es liest die
JSONL-Logs der Treiberskripte und ruft `twinlink_ml.evaluation.plots` auf. Wer eine
Abbildung ändern will, ändert das Modul, nicht diese Zelle; sonst existieren zwei
Definitionen derselben Kurve.

Erzeugt werden die Logs mit:

```bash
python scripts/study_variance_sources.py  --log artifacts/runs/variance.jsonl
python scripts/study_noise_level.py       --log artifacts/runs/noise.jsonl
python scripts/study_validation_policy.py --log artifacts/runs/policy.jsonl
```

Jede Lernkurve trägt vier Elemente, und keines davon ist Dekoration:

1. **Median über die Replikate** mit Min/Max-Band (nicht ±1σ — bei drei Replikaten ist
   die Normalverteilungsannahme nicht prüfbar).
2. **Persistenz als waagerechte Linie.** Sie wird nicht trainiert; über
   $n_\mathrm{train}$ ist sie konstant.
3. **Auflösungsschwelle** als rotes Band. Alles darin ist kein Ergebnis.
4. **Rauschgrenze** als untere Asymptote — die Höhe, bei der eine Kurve abflachen *muss*.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from twinlink_ml.evaluation.plots import (
    plot_learning_curve,
    plot_noise_study,
    plot_variance_decomposition,
)
from twinlink_ml.evaluation.resolution import (
    resolution_report,
    resolution_thresholds,
    select,
)
from twinlink_ml.utils import read_jsonl

RUNS = Path("..") / "artifacts" / "runs"
HORIZON = 24
TARGETS = ("energy_target", "product_moisture_kg_per_kg", "d50_um")

variance = read_jsonl(RUNS / "variance.jsonl")
print(len(variance), "Einträge")

## 1 Streuung nach Quelle

`data` = Datensatz-Realisierung (Anregung und Anfangszustände), `noise` = Sensorrauschen,
`model` = Initialisierung und Batch-Reihenfolge, `joint` = alle drei gemeinsam über einen
Master-Seed.

Für die Baselines ist `model` exakt null — sie haben keine stochastische Anpassung. Das
ist die Regressionsprüfung: wäre der Balken nicht null, steckte irgendwo Zufall, der dort
nicht hingehört.

In [ ]:
fig, axes = plt.subplots(len(TARGETS), 1, figsize=(9, 11), constrained_layout=True)
for axis, target in zip(axes, TARGETS, strict=True):
    plot_variance_decomposition(variance, target=target, horizon=HORIZON, ax=axis)
fig

In [ ]:
for target in TARGETS:
    print(resolution_report(variance, target=target, horizon=HORIZON))
    print()

## 2 Lernkurven mit Auflösungsschwelle

Die Schwelle stammt aus der Seed-Streuung der **parameterfreien** Persistenz und ist
deshalb unabhängig von $n_\mathrm{train}$ — sie ist eine untere Schranke des Vergleichs.
Die zusätzliche Schätzstreuung bei kleinem $n$ steht in der Tabelle oben unter `static`
und `arx`.

In [ ]:
thresholds = {
    target: resolution_thresholds(variance, target=target, horizon=HORIZON) for target in TARGETS
}

fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 4.4), constrained_layout=True)
for axis, target in zip(axes, TARGETS, strict=True):
    plot_learning_curve(
        variance,
        target=target,
        horizon=HORIZON,
        resolution=thresholds[target],
        ax=axis,
        title=target,
    )
fig

## 3 Rauschniveau und Sättigungshöhe

Die grüne Linie ist die **modellfreie** Rauschgrenze: der Fehler, den ein perfekter
Prädiktor auf denselben Fenstern erreicht. Der Abstand zwischen Kurve und Grenze ist
alles, was ein besseres Modell überhaupt noch holen kann.

In [ ]:
noise = read_jsonl(RUNS / "noise.jsonl")
levels = sorted({float(r.params["noise/relative_std"]) for r in noise})
print("Rauschniveaus:", levels)

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 4.4), constrained_layout=True)
for axis, target in zip(axes, TARGETS, strict=True):
    plot_noise_study(noise, {}, target=target, horizon=HORIZON, n_train=64, ax=axis)
fig

Die Rauschgrenzen selbst stehen nicht im Records-Log — sie sind keine
Modelleigenschaft und wären unter einem Modellnamen falsch einsortiert.
`run_noise_study` gibt sie als zweiten Rückgabewert zurück; die Zahlen stehen in
`04_stand_ap1.md`, Befund 13.

## 4 Lernkurve je Rauschniveau

Vor dem Zerlegen **filtern**: liegen zwei Rauschniveaus im selben Aufruf, wäre die
„Streuung über Seeds" in Wahrheit der Effekt des Rauschniveaus. `collect_spreads` erkennt
das und wirft — `select` ist der richtige Weg.

In [ ]:
fig, axes = plt.subplots(1, len(levels), figsize=(16, 4.4), constrained_layout=True)
for axis, level in zip(axes, levels, strict=True):
    subset = select(noise, **{"noise/relative_std": level})
    plot_learning_curve(
        subset,
        target="product_moisture_kg_per_kg",
        horizon=HORIZON,
        ax=axis,
        title=f"Produktfeuchte, sigma_rel = {level}",
    )
fig

## 5 Validierungs-Betriebsart

`FIXED_POOL` gegen `NONE`. Für alle drei Baselines ist die Differenz exakt null — keine
benutzt den Validierungssplit. Eine Differenz ungleich null wäre der Beweis, dass
Validierungsläufe ins Training sickern; die inhaltliche Zahl entsteht mit dem ersten
früh abbrechenden Modell in Schritt 6.

In [ ]:
from twinlink_ml.evaluation import summarise_policy_gap

policy = read_jsonl(RUNS / "policy.jsonl")
gaps = summarise_policy_gap(policy, target="energy_target", horizon=HORIZON)
for (label, n_train), (fixed, none, delta) in gaps.items():
    print(f"{label:>12} n={n_train:>3}  FIXED_POOL {fixed:.4f}  NONE {none:.4f}  D {delta:+.2e}")

## 6 Architekturvergleich und Beobachtbarkeit

Aus `scripts/study_architectures.py`. **Zwei Diagramme, nicht eines**: die Primärmetrik
unterscheidet sich zwischen den Betriebsarten, weil ohne Inline-Messung das Niveau des
Laufs Teil der Aufgabe ist (siehe `evaluation.protocol.primary_metric`). Ein gemeinsames
Diagramm hätte zwei verschiedene y-Achsen unter einem Namen.

In [ ]:
from twinlink_ml.evaluation import ObservationMode, primary_metric
from twinlink_ml.evaluation.resolution import architecture_verdicts

architectures = read_jsonl(RUNS / "architectures.jsonl")

for mode in (ObservationMode.INLINE, ObservationMode.SOFT_SENSOR):
    subset = select(architectures, **{"observation/mode": mode.value})
    metric = primary_metric(mode)
    reference = "persistence" if mode is ObservationMode.INLINE else "climatology"
    fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 4.4), constrained_layout=True)
    for axis, target in zip(axes, TARGETS, strict=True):
        plot_learning_curve(
            subset,
            target=target,
            horizon=HORIZON,
            metric=metric,
            resolution=resolution_thresholds(
                subset,
                target=target,
                horizon=HORIZON,
                reference_label=reference,
                metric=metric,
            ),
            ax=axis,
            title=f"{target} ({mode.value})",
        )
    display(fig)
    plt.close(fig)

Die Verdikte kommen bewusst nicht als nackte Prozentzahl: `architecture_verdicts`
liefert die Differenz **nur zusammen mit** der Auflösungsschwelle und formuliert das
Ergebnis so, wie es berichtet werden darf.

In [ ]:
for mode in (ObservationMode.INLINE, ObservationMode.SOFT_SENSOR):
    subset = select(architectures, **{"observation/mode": mode.value})
    metric = primary_metric(mode)
    free = "persistence" if mode is ObservationMode.INLINE else "climatology"
    reference = "arx" if mode is ObservationMode.SOFT_SENSOR else free
    print(f"--- {mode.value}: product_moisture gegen {reference} ---")
    for verdict in architecture_verdicts(
        subset,
        target="product_moisture_kg_per_kg",
        horizon=HORIZON,
        reference_label=reference,
        resolution_label=free,
        metric=metric,
    ):
        print(f"  n={verdict.n_train:>3}  {verdict.statement}")